# 🚀 Complete X/Twitter Scraper & Analyzer

**One-Stop Solution for X/Twitter Data Collection and Analysis**

This notebook provides everything you need to:
- ✅ Scrape X/Twitter data (tweets, users, followers)
- 📊 Convert data to pandas DataFrames
- 📈 Analyze and visualize your results
- 💾 Export data in multiple formats

**⚠️ Important Notes:**
- Twitter scraping requires accounts to be configured
- Some features may be rate-limited
- Respect Twitter's terms of service and robots.txt
- Use responsibly and ethically

---


## 📦 1. Setup & Dependencies

Install required packages and set up the environment.


In [ ]:
# Install required packages
!pip install twscrape pandas matplotlib seaborn nest-asyncio plotly

# Import libraries
import asyncio
import json
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional, Union
from contextlib import aclosing
import nest_asyncio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

# Enable nested event loops for Colab
nest_asyncio.apply()

# Set up plotting styles
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Import twscrape
from twscrape import API, gather
from twscrape.logger import set_log_level

# Set log level
set_log_level("INFO")

# Global variables for data storage
scraped_data = {}
dataframes = {}

print("✅ Complete setup finished!")
print("🔧 Libraries loaded and configured")
print("📊 Ready for data collection and analysis")


## 🔑 2. Account Management

Configure your Twitter accounts for scraping.


In [ ]:
class TwitterScraper:
    def __init__(self):
        self.api = API()

    async def check_accounts(self):
        """Check available accounts"""
        try:
            accounts = await self.api.pool.get_all()
            if not accounts:
                return False, "No accounts configured"

            active_accounts = [acc for acc in accounts if acc.active]
            inactive_accounts = [acc for acc in accounts if not acc.active]

            return len(active_accounts) > 0, f"{len(active_accounts)} active accounts available"
        except Exception as e:
            return False, f"Error checking accounts: {e}"

    async def search_tweets(self, query: str, limit: int = 100) -> List[Dict[str, Any]]:
        """Search for tweets using a query"""
        tweets = []
        try:
            async with aclosing(self.api.search(query, limit=limit)) as gen:
                async for tweet in gen:
                    tweet_data = {
                        'id': tweet.id,
                        'url': tweet.url,
                        'date': tweet.date.isoformat() if tweet.date else None,
                        'content': tweet.rawContent,
                        'username': tweet.user.username,
                        'display_name': tweet.user.displayname,
                        'user_id': tweet.user.id,
                        'verified': getattr(tweet.user, 'verified', False),
                        'followers_count': getattr(tweet.user, 'followersCount', 0),
                        'following_count': getattr(tweet.user, 'followingCount', getattr(tweet.user, 'friendsCount', 0)),
                        'reply_count': getattr(tweet, 'replyCount', 0),
                        'retweet_count': getattr(tweet, 'retweetCount', 0),
                        'like_count': getattr(tweet, 'likeCount', 0),
                        'quote_count': getattr(tweet, 'quoteCount', 0),
                        'view_count': getattr(tweet, 'viewCount', 0),
                        'lang': getattr(tweet, 'lang', ''),
                        'source': getattr(tweet, 'sourceLabel', ''),
                        'hashtags': [getattr(tag, 'text', str(tag)) for tag in getattr(tweet, 'hashtags', []) or []],
                        'mentioned_users': [user.username for user in getattr(tweet, 'mentionedUsers', []) or []],
                        'links': [{
                            'text': getattr(link, 'text', ''),
                            'url': getattr(link, 'url', ''),
                            'type': getattr(link, 'type', 'unknown')
                        } for link in getattr(tweet, 'links', []) or []],
                        'media': [{
                            'type': getattr(media, 'type', 'unknown'),
                            'url': getattr(media, 'url', ''),
                            'alt': getattr(media, 'alt', '')
                        } for media in (tweet.media if isinstance(tweet.media, list) else [tweet.media] if tweet.media else [])]
                    }
                    tweets.append(tweet_data)
                    if len(tweets) % 10 == 0:
                        print(f"📝 Collected {len(tweets)}/{limit} tweets")
        except Exception as e:
            print(f"❌ Error searching tweets: {e}")

        return tweets

    async def get_user_info(self, username: str) -> Dict[str, Any]:
        """Get detailed user information"""
        try:
            user = await self.api.user_by_login(username)
            return {
                'id': user.id,
                'username': user.username,
                'display_name': user.displayname,
                'description': getattr(user, 'rawDescription', ''),
                'verified': getattr(user, 'verified', False),
                'protected': getattr(user, 'protected', False),
                'followers_count': getattr(user, 'followersCount', 0),
                'following_count': getattr(user, 'followingCount', getattr(user, 'friendsCount', 0)),
                'statuses_count': getattr(user, 'statusesCount', 0),
                'favourites_count': getattr(user, 'favouritesCount', 0),
                'listed_count': getattr(user, 'listedCount', 0),
                'media_count': getattr(user, 'mediaCount', 0),
                'location': getattr(user, 'location', ''),
                'url': getattr(user, 'url', ''),
                'joined_date': user.created.isoformat() if getattr(user, 'created', None) else None,
                'profile_banner_url': getattr(user, 'profileBannerUrl', ''),
                'profile_image_url': getattr(user, 'profileImageUrl', '')
            }
        except Exception as e:
            print(f"❌ Error getting user info for {username}: {e}")
            return {}

    async def get_user_tweets(self, username: str, limit: int = 50) -> List[Dict[str, Any]]:
        """Get recent tweets from a user"""
        try:
            user = await self.api.user_by_login(username)
            tweets = await gather(self.api.user_tweets(user.id, limit=limit))
            return [{
                'id': tweet.id,
                'url': tweet.url,
                'date': tweet.date.isoformat() if tweet.date else None,
                'content': tweet.rawContent,
                'reply_count': tweet.replyCount,
                'retweet_count': tweet.retweetCount,
                'like_count': tweet.likeCount,
                'view_count': tweet.viewCount
            } for tweet in tweets]
        except Exception as e:
            print(f"❌ Error getting tweets for {username}: {e}")
            return []

    async def get_followers(self, username: str, limit: int = 100) -> List[Dict[str, Any]]:
        """Get followers of a user"""
        try:
            user = await self.api.user_by_login(username)
            followers = await gather(self.api.followers(user.id, limit=limit))
            return [{
                'id': follower.id,
                'username': follower.username,
                'display_name': follower.displayname,
                'verified': getattr(follower, 'verified', False),
                'followers_count': getattr(follower, 'followersCount', 0),
                'following_count': getattr(follower, 'followingCount', getattr(follower, 'friendsCount', 0)),
                'description': getattr(follower, 'rawDescription', '')
            } for follower in followers]
        except Exception as e:
            print(f"❌ Error getting followers for {username}: {e}")
            return []

# Initialize scraper
scraper = TwitterScraper()

# Check account status
accounts_ok, message = await scraper.check_accounts()
print(f"🔍 Account Status: {message}")

if not accounts_ok:
    print("\n📋 To set up accounts:")
    print("1. Create an accounts.txt file with your Twitter credentials")
    print("2. Format: username:password:email:email_password")
    print("3. Upload it to Colab or create it here")
    print("\nExample accounts.txt content:")
    print("your_username:your_password:your_email@gmail.com:email_password")
else:
    print("✅ Ready to scrape!")
